# Derivative of the matter power spectrum

The smallest use of DISCO-EB's derivatives: $\partial P(k) / \partial \Omega_m$ today by forward-mode automatic differentiation, checked against a finite difference.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from discoeb import LCDM, matter_power_spectrum, solve_eb

K = np.geomspace(1.0e-4, 1.0, 64)  # wave numbers, 1/Mpc

## The cosmology

The Planck 2018 best fit, flat, with three massless neutrino flavours, given by $\Omega_m$, $\Omega_b$ and $H_0$. A model's own parameters are the physical densities $\omega = \Omega h^2$, and `from_density_parameters` converts, finding the dark-energy density that makes the model expand at $H_0$ today.

In [2]:
h = 0.6732117
PLANCK = dict(
    Omegam=(0.02238280 + 0.1201075) / h**2,
    Omegab=0.02238280 / h**2,
    H0=100.0 * h,
    A_s=2.100549e-9,
    n_s=0.9660499,
    Tcmb=2.7255,
    YHe=0.2454006,
    Neff=3.046,
)

## From $\Omega_m$ to $P(k)$

`power` maps $\Omega_m$ to $P(k)$ today. `from_density_parameters` is plain JAX arithmetic, so the derivative goes through it. `solve_eb` integrates in the scale factor and keeps the two density contrasts at `a = 1`, today, and `matter_power_spectrum` turns them into $P(k)$.

$\Omega_b$, $H_0$ and flatness are held, so more matter means less dark energy.

In [3]:
def power(Omegam):
    """``P(k)`` today in Mpc^3, ``(len(K),)``, of the Planck cosmology at this Omega_m."""

    cosmology = LCDM.from_density_parameters(**{**PLANCK, "Omegam": Omegam})
    values = solve_eb(
        cosmology, K, [1.0], keep=("clxc", "clxb")
    )  # (cosmology, name, k, a)
    clxc, clxb = jnp.unstack(values[0, :, :, -1])
    return matter_power_spectrum(
        K,
        clxc=clxc,
        clxb=clxb,
        omega_c=cosmology.omega_c,
        omega_b=cosmology.omega_b,
        A_s=cosmology.A_s,
        n_s=cosmology.n_s,
        k_p=cosmology.k_p,
    )

## The derivative

`jax.jacfwd` of `power` gives $\partial P / \partial \Omega_m$ at every wave number from one forward-sensitivity launch. The solve differentiates at fixed scale factor, so this is the derivative of $P(k)$ today with nothing to add.

In [4]:
Omegam = jnp.asarray(PLANCK["Omegam"])
pk = power(Omegam)
dpk = jax.jacfwd(power)(Omegam)
slope = np.asarray(Omegam * dpk / pk)  # d ln P / d ln Omega_m

## The check

A central finite difference of two more solves.

In [5]:
step = 1.0e-3
finite = np.asarray(
    Omegam * jnp.log(power(Omegam + step) / power(Omegam - step)) / (2.0 * step)
)

The two slopes agree to better than $10^{-2}$.

In [6]:
print(
    f"{'k [1/Mpc]':>12}  {'P(k) [Mpc^3]':>14}  {'dlnP/dlnOm':>11}  {'finite diff.':>12}"
)
for i in range(0, len(K), 7):
    print(f"{K[i]:12.4e}  {pk[i]:14.6e}  {slope[i]:11.5f}  {finite[i]:12.5f}")
print(f"largest difference between the two slopes: {np.abs(slope - finite).max():.1e}")

   k [1/Mpc]    P(k) [Mpc^3]   dlnP/dlnOm  finite diff.
  1.0000e-04    2.017416e+03     -1.53710      -1.53695
  2.7826e-04    5.402935e+03     -1.53109      -1.53117
  7.7426e-04    1.421702e+04     -1.49816      -1.49836
  2.1544e-03    3.486640e+04     -1.37569      -1.37609
  5.9948e-03    6.883243e+04     -1.03788      -1.03743
  1.6681e-02    7.392886e+04     -0.16225      -0.16094
  4.6416e-02    3.258177e+04      0.74347       0.74307
  1.2915e-01    7.019365e+03      1.50651       1.50601
  3.5938e-01    9.229104e+02      2.13959       2.13900
  1.0000e+00    9.044886e+01      2.42823       2.42799
largest difference between the two slopes: 2.3e-03
